# 비산란 복사 전달 모델링

이 노트북은 Albany 대학교의 [Brian E. J. Rose](https://www.atmos.albany.edu/facstaff/brose/)가 제공하는 [The Climate Laboratory](https://brian-rose.github.io/ClimateLaboratoryBook)의 일부입니다.

## 1. 두 흐름 슈바르츠실트 방정식

여기서는 복사 전달의 지배 방정식을 소개하여, 우리가 지금까지 다뤄온 내용들을 보다 탄탄한 이론적 토대 위에 세우고자 합니다.

여기서의 유도 과정은 또한 `climlab`에 일부 복사 솔버(radiation solver)가 어떻게 구현되는지에 대한 체계적인 문서 역할을 할 것입니다.

### 광학적 두께

흡수체 층의 **광학적 두께**는 $\Delta \tau_\nu$입니다. 이는 대기층의 방출률(emissivity)과 흡수율(absorptivity)을 나타냅니다.

매우 얇은 층의 극한(limit)으로 가면, 우리는 다음 식을 통해 $\tau_\nu$를 정의합니다:

$$ \frac{d \tau_\nu}{dp} = -\frac{1}{g} \kappa_\nu $$

여기서 **$\kappa_\nu$는 주파수 $\nu$에서의 단위 질량당 흡수 단면적(absorption cross-section)**입니다. 그 단위는 m$^2$ kg$^{-1}$입니다. $\kappa$는 대기 단위 질량 내 흡수체들이 입사 빔(incident beam)에서 제거하는 면적을 측정하는 척도입니다.

일반적으로 **$\tau_\nu$는 복사(radiation)의 주파수(frequency)에 따라 달라지며**, 이는 여기에서 아래 첨자 $\nu$로 나타나듯 합니다.

### 광학 깊이(optical depth)를 수직 좌표(vertical coordinate)로 활용

고도가 높아질수록 압력(pressure)이 감소하므로, $\tau_\nu$는 고도에 따라 증가합니다.

복사 전달 방정식(radiative transfer equations)은 압력(pressure) 대신 $\tau_\nu$를 수직 좌표(vertical coordinate)로 사용함으로써 단순화될 수 있습니다.

### 흡수 단면적

비흡수 단면적 $\kappa$는 복사선이 통과하면서 마주치는 각 온실가스 분자의 수와 각 온실가스 분자 종류의 고유한 흡수 특성에 따라 결정됩니다. $q_k$를 $k$번째 온실가스의 질량-비 농도(mass-specific concentration)라고 하면, 일반적으로 다음과 같이 쓸 수 있습니다.

$$ \kappa(\nu, p, T) = \sum_{k=1}^n \kappa_k\big(\nu, p, T \big) q_k(p) $$

잘 혼합된 온실가스의 경우, $q_k$는 상수입니다. 수증기처럼 잘 혼합되지 않은 기체의 경우, 우리는 $q_k(p)$를 통해 해당 기체의 수직 분포를 고려해야 합니다.

$\kappa_k$가 온도와 압력에 의존하는 현상은 분자 흡수(molecular absorption) 물리학의 특정 측면에서 기인합니다.

### 두 흐름 슈바르츠실트 방정식

기후 모델링에서 우리는 총 복사속(flux)을 거의 항상 두 흐름(beam)으로 나눕니다: **상향** 및 **하향**.

이 과정은 복사속의 전체 각도 의존성에 대한 적분을 계산하는 것을 수반합니다. 여기서는 자세한 내용은 생략하겠습니다.

$U_\nu$를 상향 복사선으로, $D_\nu$를 하향 복사선으로 둡니다. 이 복사선들에 대한 지배 방정식은 다음과 같은 *슈바르츠실트 방정식(Schwarzschild equations)*입니다:

$$
\begin{align}
\frac{d U_\nu}{d \tau_\nu} &= -U_\nu + E\big( \nu, T(\tau_\nu) \big) \\
\frac{d D_\nu}{d \tau_\nu} &= D_\nu - E\big( \nu, T(\tau_\nu) \big) 
\end{align}
$$

여기서 $E$는 (상향 및 하향 모두) **흑체 방출(blackbody emission)**이며, 일반적으로 **주파수(frequency)**와 **온도(temperature)** 모두에 의존합니다. 우리는 온도를 연직 좌표인 광학적 깊이(optical depth)의 함수로 표현했습니다.

방출량은 **플랑크 함수(Planck function)**에 의해 결정됩니다:

$$
\begin{align}
E &= \pi~ B\big( \nu, T \big) \\
B\big( \nu, T \big) &= \frac{2 h \nu^3}{c^2} \frac{1}{\exp \left( \frac{h \nu}{k T} \right) -1}
\end{align}
$$

다음은 이러한 기본 물리 상수들입니다:

- $h = 6.626 \times 10^{-34} ~\text{J s} $는 플랑크 상수(Planck's constant)입니다.
- $c = 3.00 \times 10^8 ~\text{m s}^{-1} $는 빛의 속도(speed of light)입니다.
- $k = 1.38 \times 10^{-23} ~\text{J K}^{-1} $는 볼츠만 열역학 상수(Boltzmann Thermodynamic Constant)입니다.

이류 방정식은 기본적으로 복사선이 각 얇은 기체층에서 **흡수에 의해 감쇠되고** (첫 번째 항) **방출에 의해 증강된다** (두 번째 항)는 것을 설명합니다.

이 방정식들은 **산란의 영향을 받지 않는** 복사선에 대해 유효합니다. 이 내용은 나중에 다시 다룰 수 있습니다.

## 2. 회색 기체 모델

대부분의 대기 가스의 흡수 특성($\kappa_i$)은 복사(선)의 주파수에 따라 엄청나게 달라집니다. 따라서 광학적 두께($\tau_\nu$) 또한 파수(wavenumber) 또는 주파수에 복잡하게 의존합니다.

기후 모델러로서 우리의 임무는 대기의 순 복사(선) 흡수 및 투과를 계산하고 이해하는 것입니다. 이를 철저하고 정확하게 수행하려면, 복사 플럭스(fluxes)를 매우 조밀한 파수(wavenumber) 격자에서 개별적으로 계산한 다음, 그 결과를 모든 파수에 걸쳐 적분해야 합니다.

전지구 기후 모델(GCM)에 사용되는 실제 복사 전달 코드들은 이러한 무차별 대입 방식(brute-force approach)을 단순화하기 위해 많은 기법과 단축 과정을 적용하지만, 결과적으로 이해하기 어려운 일련의 방정식으로 귀결됩니다.

그러나 우리는 플럭스(flux)의 스펙트럼 의존성(spectral dependence)을 무시함으로써 복사 전달(radiative transfer)과 온실 효과(greenhouse effect)의 기본에 대해 많은 것을 이해할 수 있습니다.

구체적으로, 우리는 다음 근사를 사용합니다.

$$ \kappa(\nu, p, T) = \kappa(p) $$

따라서 광학적 깊이($\tau$)는 이제 **주파수와 무관**하게 됩니다.

이는 **회색 기체 근사(grey gas approximation)**라고 알려져 있습니다.

## 회색 기체 형태의 슈바르츠실트 방정식

### 플랑크 함수 적분

만약 우리가 $\tau_\nu$가 진동수(frequency)에 독립적이라고 가정한다면, 모든 진동수에 걸쳐 두 흐름 방정식(two-stream equations)을 적분할 수 있습니다.

플랑크 함수의 적분은 우리가 잘 알고 있는 슈테판-볼츠만 흑체 복사 법칙(Stefan-Boltzmann blackbody radiation law)을 제공합니다.

$$ E(T) = \int_0^{\infty} \pi B\big( \nu, T \big) d\nu = \sigma T^4  $$

여기서

$$ \sigma = \frac{2 \pi^5 k^4}{15 c^2 h^3} = 5.67 \times 10^{-8} ~\text{W m}^{-2}~\text{K}^{-4}$$

In [1]:
# climlab has these constants available, and actually calculates sigma from the above formula
import numpy as np
import climlab 

sigma = ((2*np.pi**5 * climlab.constants.kBoltzmann**4) / 
         (15 * climlab.constants.c_light**2 * climlab.constants.hPlanck**3) )
print( sigma)
sigma == climlab.constants.sigma

5.6703726225913323e-08


True

### 회색 기체 방정식

다음은 회색 기체 모델의 지배 방정식을 나타냅니다:

$$
\begin{align}
\frac{d U}{d \tau} &= -U + \sigma T(\tau)^4 \\
\frac{d D}{d \tau} &= D - \sigma T(\tau)^4 
\end{align}
$$

이 방정식들은 복사 흐름(beam)이 얇은 층에서의 흡수(첫 번째 항)에 의해 감소하고, 흑체 복사 방출(blackbody emission)에 의해 증가하며, 이 두 과정 모두 주파수(frequency)에 독립적이라고 가정한다는 것을 의미합니다.

### 흡수 물질의 유한한 층을 통과하면서 플럭스(flux)는 어떻게 변하는가?

위 방정식들은 **선형 1차 상미분 방정식(ODEs)**이며, 서로 비결합되어 있습니다 (이는 우리가 산란을 무시했기 때문입니다).

광학 깊이(optical level) $\tau_{0}$부터 $\tau_{1}$까지의 **대기층 하나를 생각해 봅시다**. 해당 층의 광학 두께(optical thickness)는 $\Delta \tau = \tau_{1} - \tau_{0}$ 입니다.

아래로부터 입사하는 상향 복사(upwelling beam)는 $U_{0}$로 표시되며; 우리 층의 상단에서 나가는 상향 복사는 $U_{1}$로 표시됩니다. 우리는 $U_{1}$을 계산하고자 하며, 이는 우리 층에 대해 $dU/d\tau$ 방정식을 적분하여 얻을 수 있습니다.

그 결과는 다음과 같습니다.

$$ U_{1} = U_{0} \exp(-\Delta \tau) + \int_{\tau_{0}}^{\tau_{1}} E \exp \big( -(\tau_{1} - \tau) \big) d\tau $$

여기서 $\tau$는 이제 적분의 더미 변수(dummy variable of integration)이며, 흑체 방출(blackbody emissions)은 $E = \sigma T^4$로 나타냈습니다.

하향 복사(downwelling radiation)의 변화도 유사합니다:

$$ D_{0} = D_{1} \exp(-\Delta \tau ) + \int_{\tau_0}^{\tau_1} E \exp\big( -(\tau - \tau_{0}) \big) d\tau $$

## 3. 유한 격자 위에서 회색기체 방정식의 이산화

유한한 층(finite layers)에 걸친 이러한 변화들을 고려하는 특히 중요한 이유 중 하나는, 우리가 일반적으로 온도가 이산 격자(discrete grid, 종종 압력 좌표를 사용)에 표현되는 수치 모델(numerical model)에서 복사 전달 방정식(radiative transfer equations)을 풀기 때문입니다.

만약 **우리 층 전체에 걸쳐 온도가 균일하다고** 가정한다면, 흑체 복사(blackbody emission) $E$ 또한 층 전체에 걸쳐 일정하며, 이를 $E_0$로 표기하겠습니다.

이는 적분 밖으로 나올 수 있으며, 그 표현식들은 다음과 같이 단순화됩니다.

$$
\begin{align}
U_1 &= U_0 ~ \exp(-\Delta \tau )  + E_{0} ~\Big( 1 - \exp(-\Delta \tau) \Big)  \\
D_0 &= D_1 ~ \exp(-\Delta \tau )  + E_{0} ~\Big( 1 - \exp(-\Delta \tau) \Big)
\end{align}
$$

### 투과율 (Transmissivity)
첫 번째 항은 층의 하단에서 상단으로 (또는 그 반대로) 복사(Radiation)의 **투과 (Transmission)**를 의미합니다. 우리는 층의 **투과율 (Transmissivity)** ( $t_{0}$로 표기)을 입사 빔(Incident beam) 중 다음 층으로 전달되는 부분의 비율로 정의할 수 있습니다.

$$
\begin{align}
t_{0} &= \frac{U_{0} \exp(-\Delta \tau)}{U_{0}}  \\
 &= \exp(-\Delta \tau)\\
\end{align}
$$

### 방출률 (Emissivity)
두 번째 항은 해당 층 내 **방출**로 인한 복사 빔의 순 변화입니다.

우리는 유사하게 해당 층의 **방출률** $\epsilon_0$을 다음과 같이 정의할 수 있습니다.

\begin{align}
\epsilon_{0} &=  1 - \exp(-\Delta \tau) \\
  &= 1 - t_{0} 
\end{align}

### 유한 압력 격자에서의 이산 이중 흐름 방정식

이 모든 내용을 종합하면, 광학 깊이(optical depth) $\Delta \tau$를 갖는 이산적인 층을 통과하는 상향 복사(upwelling) 빔과 하향 복사(downwelling) 빔의 변화를 나타내는 두 가지 간단한 방정식을 얻을 수 있습니다.

$$
\begin{align}
U_{1} &= t_0 ~ U_{0} + \epsilon_{0} ~ E_{0}  \\
D_{0} &= t_0 ~ D_{1} + \epsilon_{0} ~ E_{0}  \\
\end{align}
$$

저희 모델은 일반적으로 압력 좌표계에서 이산화됩니다. 만약 층의 두께가 $\Delta p$라면, 광학 깊이(optical depth)는 다음과 같습니다.

$$ \Delta \tau = -\frac{\kappa}{g} \Delta p$$
(음의 부호는 두 좌표계의 반대되는 부호 규약(sign conventions)을 반영합니다.)

따라서 해당 층의 방출률은 다음과 같습니다.

$$  \epsilon_{0} = 1 - \exp\big( \frac{\kappa}{g} \Delta p \big)  $$

회색 기체 근사(grey gas approximation)에서는 복사(flux)의 모든 스펙트럼 의존성을 무시하므로, $\kappa$는 상수입니다 (수증기처럼 잘 혼합되지 않은 온실 가스를 나타내기 위해 수직적으로 변화하도록 둘 수도 있습니다). 그리고 흑체 복사(blackbody emissions)는 단순히 다음과 같습니다.

$$ E_0 = \sigma T_0^4 $$

## 4. 회색 기체 모델에 대한 행렬 방정식

`climlab`은 위에서 기술된 바와 같이 이산화된 두 흐름 방정식을 구현합니다.

이제 $N$개의 층으로 구성된 이산화된 압력 격자에서 방정식들을 작성해 보겠습니다.

**상향 플럭스(upwelling flux)**를 다음 열 벡터라고 하겠습니다.

$${\bf{U}} = [U_0, U_1, ..., U_{N-1}, U_N]^T$$

$N$개의 층(level)이 있다면 $\bf{U}$는 $N+1$개의 원소를 가집니다 (즉, 플럭스(flux)는 층(level) 사이의 경계에서 정의됩니다). `numpy` 인덱스(index) 규칙에 따라 층에 0부터 번호를 매기겠습니다.

- $U_0$는 지표면에서 0번 층으로의 상향 플럭스(upwelling flux)입니다.
- $U_1$은 0번 층에서 1번 층으로의 상향 플럭스 등입니다.
- $U_N$은 N-1번 층(가장 위 층)에서 우주로의 상향 플럭스입니다.

**하향 플럭스(downwelling flux)**도 마찬가지입니다.

$${\bf{D}} = [D_0, D_1, ..., D_N]^T$$

따라서 $D_N$은 우주에서 내려오는 플럭스이고 $D_0$는 지표면으로의 역복사(back-radiation)입니다.

각 $N$개의 기압층에 대해 **온도**와 **흑체 방출량(blackbody emissions)**이 정의되며, 이들은 다음 식에 의해 서로 관련됩니다:

$$E_i = \sigma T_i^4$$

### 방출률 및 투과율 벡터 (Emissivity and Transmissivity Vectors)

흡수율(absorptivity)/방출률(emissivity) 벡터를 다음과 같이 정의합니다.

$$ {\bf{\epsilon}} = [\epsilon_0, \epsilon_1, ..., \epsilon_{N-1}] $$

여기서 각 요소는 해당 층의 두께와 이 특정 스펙트럼 대역(spectral band)에 대한 흡수 단면적(absorption cross-section)에 의해 결정됩니다.

$$ \epsilon_{i} = 1 - \exp\big( \frac{\kappa}{g} \Delta p_i \big) $$

그리고 개별 층의 투과율(transmissivity)은 다음과 같습니다.

$$ t_{i} = 1 - \epsilon_{i} $$

$N+1$개의 요소를 가지는 투과율 벡터(vector of transmissivities) ${\bf{t}}$를 정의하는 것이 편리합니다:

$$ {\bf{t}} = [1, t_0, t_1, ..., t_{N-1}] $$

### 하향 복사 빔

하향 복사 빔의 경우, 우리는 $N+1$개의 요소를 가진 방출량의 열 벡터를 정의합니다:

$$ {\bf{E_{down}}} = [E_0, E_1, ..., E_{N-1}, E_N]^{T} $$

여기서 우리는 마지막 요소 $E_N$을 **TOA(대기 상단)에서의 입사 플럭스(incident flux)**로 정의합니다.

장파 모델의 경우, 우리는 일반적으로 $E_N = 0$으로 설정합니다. 단파 모델의 경우, 이것은 입사 태양 복사(incident solar radiation)가 됩니다.

우리는 $E_N$과 곱해질 때, 각 $N+1$개 층 경계면에서 하향 복사 빔(downwelling beam)을 제공하는 행렬 ${\bf{T_{down}}}$을 원합니다.

$${\bf{D}} = {\bf{T_{down}}} ~ {\bf{E_{down}}} $$

### 상향 복사(upwelling beam)

상향 복사(upwelling beam)의 방출량 벡터를 다음과 같이 정의합니다:

$$ {\bf{E_{up}}} = [up_{sfc}, E_0, E_1, ..., E_{N-1}] $$

우리는 지표면으로부터의 어떠한 방출량에도 지표면에 도달하는 하향 복사(downwelling beam) 중 반사된 부분을 더해야 합니다.

$$ up_{sfc} = E_{sfc} + \alpha_{sfc} ~ D[0] $$

이제 우리는 상향 복사빔(upwelling beam)이 다음과 같이 표현되도록 하는 행렬 ${\bf{T_{up}}}$를 찾고자 합니다:

$${\bf{U}} = {\bf{T_{up}}} ~ {\bf{E_{up}}} $$

### 투과율 행렬

$$ {\bf{T_{up}}} = \left[ \begin{array}{cccccc} 1 & 0 & 0 & ... & 0 & 0 \\  
                                             t_0 & 1 & 0 & ... & 0 & 0 \\
                                         t_1 t_0 & t_1 & 1 & ... & 0 & 0 \\
                                     t_2 t_1 t_0 & t_2 t_1 & t_2 & ... & 0 & 0 \\
                                     ... & ... & ... & ... & ... & ... \\
 \prod_0^{N-1} t_i & \prod_1^{N-1} t_i & \prod_2^{N-1} t_i & ... & t_{N-1} & 1 
   \end{array} \right] $$
   
그리고
 
$$ {\bf{T_{down}}} = {\bf{T_{up}}}^T  $$

이 공식들은 벡터화된 `numpy` 배열 연산을 사용하여 `climlab.radiation.transmissivity.Transmissivity()`에 구현되었습니다.

In [2]:
# example with N=2 layers and constant absorptivity
#  we construct an array of absorptivities
eps = np.array([0.58, 0.58])
#  and pass these as argument to the Transmissivity class
trans = climlab.radiation.transmissivity.Transmissivity(eps)
print( 'Matrix Tup is ')
print( trans.Tup)
print( 'Matrix Tdown is ')
print( trans.Tdown)

Matrix Tup is 
[[1.     0.42   0.1764]
 [0.     1.     0.42  ]
 [0.     0.     1.    ]]
Matrix Tdown is 
[[1.     0.     0.    ]
 [0.42   1.     0.    ]
 [0.1764 0.42   1.    ]]


## 5. `climlab` 내 밴드 평균 복사 모델

회색 기체 모델(Grey Gas model)은 복사(radiation)가 전 지구 에너지 균형을 어떻게 형성하는지 이해하는 데 유용한 첫걸음입니다. 하지만 기후 민감도(climate sensitivity)를 실제로 무엇이 결정하는지 이해하려고 할 때, 우리는 그 한계에 빠르게 부딪히게 됩니다.

모델 계층(model hierarchy)에서 다음 단계는 무엇일까요?

우리가 스펙트럼을 $M$개의 이산적인 **스펙트럼 대역(spectral bands)**으로 나눈다고 가정해 봅시다. 이 아이디어는 주요 기체들의 흡수 특성이 비교적 균일한 스펙트럼의 부분을 찾는 것입니다. 그런 다음, $k$번째 기체와 $j$번째 대역에 대한 **대역 평균 흡수 단면적(band-averaged absorption cross section)**을 다음과 같이 표현합니다.

$$ \kappa_{kj} \big(p, T \big) = \frac{\int_{\nu_j} \kappa_j \big(\nu, p, T \big) d \nu }{  \int_{\nu_j} d \nu } $$

여기서 우리는 $j$번째 대역을 정의하기 위해 선택한 스펙트럼의 해당 부분에 대해 적분합니다.

우리의 띠 모델(band model)에서는 일반적으로 $\kappa$의 온도 의존성을 무시할 것입니다. 따라서 해당 띠의 총 흡수 단면적(total absorption cross section)은 (모든 흡수 기체를 합산하면) 다음과 같습니다:

$$ \kappa_j(p) = \sum_{k=1}^n \kappa_{kj}(p) q_k(p) $$

일단 이 정의를 내리면, 위에서 설명한 회색 기체 모델(grey gas model)의 모든 공식이 각 띠의 플럭스(flux)에 대해 거의 동일하게 적용될 수 있다는 점을 유의하십시오.

밴드 $j$에서의 광학 깊이(optical depth)는 다음과 같습니다.

$$ \Delta \tau_j = -\frac{\kappa_j}{g} \Delta p$$

이로부터 우리는 위에서와 마찬가지로 밴드 $j$에 대한 방출률(emissivity)과 투과율(transmissivity)을 정의할 수 있습니다.

회색 기체 공식(Grey Gas formulas)과의 유일한 차이점은 밴드 $j$에서 발생하는 **흑체 복사(blackbody emission)** ($E_j$로 표기)가 이제 $\sigma T^4$의 일부에 불과하다는 것입니다.

이 일부를 $b_j$로 나타내겠습니다.

분율 $b_j$는 온도에 의존하며, 플랑크 함수(Planck function)를 적분하여 풀 수 있습니다:

$$ E_j(T) = \int_{\nu_j} \pi B\big( \nu, T \big) d\nu $$

띠 모델(band model)을 단순화하기 위해, 미리 $b_j$를 고정하도록 선택할 수 있으며, 다음과 같이 가정할 수 있습니다:

$$ E_j(T) = b_j ~ \sigma ~ T^4 $$

이는 온도가 크게 변하지 않는다면 합리적입니다.

$b_j$를 어떻게 계산하든지 상관없이, 우리 모델의 모든 대역에서 $b_j$의 합은 1이 되어야 합니다:

$$ \sum_0^{M-1} b_j = 1 $$

일단 총 플럭스를 여러 밴드로 나누는 방법을 알아내고 각 밴드의 흡수 단면적을 알게 되면, **회색 기체 모델에서 사용하는 것과 동일한 공식(동일한 코드!)을 사용하여** 각 밴드별로 상향 및 하향 플럭스를 독립적으로 계산할 수 있습니다.

총 플럭스(total flux)를 얻기 위해서는, 모든 밴드(band)에 걸쳐 빔(beam)들을 합산하기만 하면 됩니다.

$$
\begin{align}
U &= \sum_0^{M-1} U_j  \\
D &= \sum_0^{M-1} D_j  
\end{align}
$$

:::{note}
복사 전달(radiative transfer)에 대한 더 많은 세부 정보와 2-흐름 방정식(two-stream equations)의 더 면밀한 유도 과정을 알고 싶다면, {cite:t}`Pierrehumbert:principles`의 저서인 *행성 기후의 원리(Principles of Planetary Climate)*를 참조하십시오.
:::

## 출처

이 노트북은 올버니 대학교(University at Albany)의 [브라이언 E. J. 로즈(Brian E. J. Rose)](https://www.atmos.albany.edu/facstaff/brose/)가 개발하고 관리하는 오픈소스 교과서인 [더 기후 연구실 (The Climate Laboratory)](https://brian-rose.github.io/ClimateLaboratoryBook)의 일부입니다.

이 자료는 [크리에이티브 커먼즈 저작자표시 4.0 국제 (CC BY 4.0)](https://creativecommons.org/licenses/by/4.0/) 라이선스에 따라 자유롭게 이용 가능합니다.

이 노트와 [climlab 소프트웨어](https://github.com/climlab/climlab)의 개발은 브라이언 로즈에게 부여된 AGS-1455071 과제 번호로 국립과학재단(National Science Foundation)으로부터 부분적으로 지원받았습니다. 여기에 표현된 모든 의견, 연구 결과, 결론 또는 권고는 전적으로 저의 견해이며, 반드시 국립과학재단(National Science Foundation)의 견해를 반영하는 것은 아닙니다.